In [1]:
#import current working directory

import os
import sys

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation\research


In [2]:
# Read the sys path
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation")

In [3]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\E_Commerce_Customer_Segmentation\\research'

In [4]:
# move to parent directory

os.chdir("../") 

In [5]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\E_Commerce_Customer_Segmentation'

In [6]:
# test project import

import box

print(box.__version__)

7.4.1


In [7]:
# entity

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    data_file: Path
    status_file: Path



In [8]:
from E_Commerce_Customer_Segmentation.constant import *
from E_Commerce_Customer_Segmentation.utils.common import read_yaml,create_directories
from E_Commerce_Customer_Segmentation.entity.config_entity import DataValidationConfig

In [9]:
# configuration manager

class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:

        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            data_file=Path(config.data_file),
            status_file=Path(config.status_file)
        )

        return data_validation_config

In [10]:
import os
import pandas as pd

from E_Commerce_Customer_Segmentation.logging import logger
from E_Commerce_Customer_Segmentation.entity.config_entity import DataValidationConfig


In [ ]:
# components


class DataValidation:

    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_dataset(self):

        validation_status = True

        # 1. DATASET EXISTENCE CHECK

        if not os.path.exists(self.config.data_file):

            logger.info("Dataset Not Found")

            return False

        logger.info("Dataset Found")

        # Read Dataset

        df = pd.read_excel(self.config.data_file)

        # 2. DATASET SHAPE CHECK

        rows, columns = df.shape

        logger.info(f"Rows : {rows}")
        logger.info(f"Columns : {columns}")

        if rows == 0:

            logger.info("Dataset is Empty")

            validation_status = False

        if columns == 0:

            logger.info("Dataset has No Columns")

            validation_status = False

        # 3. SCHEMA / EXPECTED COLUMN CHECK

        expected_columns = [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]

        actual_columns = list(df.columns)

        missing_columns = (set(expected_columns) - set(actual_columns))

        extra_columns = (set(actual_columns) - set(expected_columns))

        if not missing_columns and not extra_columns:

            logger.info("Schema Validation Passed")

        else:

            logger.info("Schema Validation Failed")

            if missing_columns:

                logger.info(f"Missing Columns : {missing_columns}")

            if extra_columns:

                logger.info(f"Extra Columns : {extra_columns}")

            validation_status = False

        # 4. DATA TYPE VALIDATION

        expected_dtypes = {
            "InvoiceNo": "object",
            "StockCode": "object",
            "Description": "object",
            "Quantity": "int64",
            "InvoiceDate": "datetime64[ns]",
            "UnitPrice": "float64",
            "CustomerID": "float64",
            "Country": "object"
        }

        actual_dtypes = df.dtypes.astype(str).to_dict()

        for column, expected_dtype in expected_dtypes.items():

            if column not in actual_dtypes:

                logger.info(f"{column} is Missing")

                validation_status = False

            elif actual_dtypes[column] != expected_dtype:

                logger.info(
                    f"{column} datatype mismatch | "
                    f"Expected: {expected_dtype} | "
                    f"Actual: {actual_dtypes[column]}"
                )

                validation_status = False

        logger.info("Data Type Validation Completed")

        # 5. MISSING VALUE VALIDATION

        missing_values = df.isnull().sum()

        logger.info("Missing Value Summary:")
        logger.info(missing_values[missing_values > 0])

        # CustomerID is important for customer segmentation.
        # Description can contain missing values.

        if df["InvoiceDate"].isnull().sum() > 0:

            logger.info("Missing InvoiceDate values found")

            validation_status = False

        if df["Country"].isnull().sum() > 0:

            logger.info("Missing Country values found")

            validation_status = False

        logger.info("Missing Value Validation Completed")

        # 6. DUPLICATE ROW CHECK

        duplicates = df.duplicated().sum()

        logger.info(f"Duplicate Rows : {duplicates}")

        if duplicates > 0:

            logger.info(
                "Duplicate rows found. "
                "These will be handled during "
                "data transformation."
            )

        else:

            logger.info("No Duplicate Rows Found")

        # 7. CUSTOMER ID VALIDATION

        missing_customer_id = df["CustomerID"].isnull().sum()

        unique_customers = df["CustomerID"].nunique()

        logger.info(f"Missing CustomerID : {missing_customer_id}")

        logger.info(f"Unique Customers : {unique_customers}")

        if unique_customers == 0:

            logger.info("No valid CustomerID values found")

            validation_status = False

        else:

            logger.info("CustomerID Validation Completed")

        # 8. QUANTITY VALIDATION

        invalid_quantity = (df["Quantity"] <= 0).sum()

        logger.info(
            f"Non-positive Quantity Rows : "
            f"{invalid_quantity}"
        )

        if invalid_quantity > 0:

            logger.info(
                "Non-positive Quantity values found. "
                "These will be handled during "
                "data transformation."
            )

        else:

            logger.info(
                "Quantity Validation Passed"
            )

        # 9. UNIT PRICE VALIDATION

        invalid_unit_price = (df["UnitPrice"] <= 0).sum()

        logger.info(
            f"Non-positive UnitPrice Rows : "
            f"{invalid_unit_price}"
        )

        if invalid_unit_price > 0:

            logger.info(
                "Non-positive UnitPrice values found. "
                "These will be handled during "
                "data transformation."
            )

        else:

            logger.info(
                "UnitPrice Validation Passed"
            )

        # 10. INVOICE DATE VALIDATION

        missing_invoice_date = (
            df["InvoiceDate"].isnull().sum()
        )

        if missing_invoice_date > 0:

            logger.info("Invalid InvoiceDate values found")

            validation_status = False

        else:

            logger.info("InvoiceDate Validation Passed")

        # 11. COUNTRY VALIDATION

        missing_country = (df["Country"].isnull().sum())

        if missing_country > 0:

            logger.info("Missing Country values found")

            validation_status = False

        else:

            logger.info("Country Validation Passed")

        # 12. WRITE VALIDATION STATUS

        os.makedirs(
            os.path.dirname(
                self.config.status_file
            ),
            exist_ok=True
        )

        with open(self.config.status_file,"w") as file:

            file.write(f"Validation Status : "f"{validation_status}")

        logger.info(f"Validation Status : "f"{validation_status}")

        return validation_status

In [12]:
# pipeline

class DataValidationTrainingPipeline:

    def __init__(self):
        pass

    def main(self):

        try:

            logger.info(">>>>>> Data Validation Stage Started <<<<<<")

            config = ConfigurationManager()

            data_validation_config = (config.get_data_validation_config())

            data_validation = DataValidation(config=data_validation_config)

            validation_status = (data_validation.validate_dataset())

            if validation_status:

                logger.info(">>>>>> Data Validation Stage Completed Successfully <<<<<<")

            else:

                logger.info(">>>>>> Data Validation Stage Failed <<<<<<")

            return validation_status

        except Exception as e:

            logger.exception(e)

            raise e

In [13]:
obj = DataValidationTrainingPipeline()
obj.main()

[2026-08-27 01:29:31,290: INFO: 439274918: >>>>>> Data Validation Stage Started <<<<<<]
[2026-08-27 01:29:31,296: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-27 01:29:31,298: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-27 01:29:31,298: INFO: common: created directory at: artifacts]
[2026-08-27 01:29:31,304: INFO: common: created directory at: artifacts/data_validation]
[2026-08-27 01:29:31,306: INFO: 910062656: Dataset Found]
[2026-08-27 01:30:20,240: INFO: 910062656: Rows : 541909]
[2026-08-27 01:30:20,252: INFO: 910062656: Columns : 8]
[2026-08-27 01:30:20,253: INFO: 910062656: Schema Validation Passed]
[2026-08-27 01:30:20,254: INFO: 910062656: Data Type Validation Completed]
[2026-08-27 01:30:20,355: INFO: 910062656: Missing Value Summary:]
[2026-08-27 01:30:20,362: INFO: 910062656: Description      1454
CustomerID     135080
dtype: int64]
[2026-08-27 01:30:20,382: INFO: 910062656: Missing Value Validation Completed]
[2026-08-27 

True